# Lab Work - 2.3


In [ ]:
# Common Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer, KNNImputer
from sklearn.linear_model import BayesianRidge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score
np.random.seed(42)

## Q1: Warm-up (Numeric Only)

In [ ]:
# Create correlated data
n = 500
X = np.random.randn(n, 5)
X[:,1] = X[:,0]*0.8 + np.random.randn(n)*0.2
X[:,2] = X[:,0]*0.5 + X[:,1]*0.3 + np.random.randn(n)*0.2
df = pd.DataFrame(X, columns=[f'X{i}' for i in range(5)])

# Introduce missing
df_missing = df.copy()
for col in ['X0','X1','X2']:
    df_missing.loc[df_missing.sample(frac=0.15).index, col] = np.nan

# Impute
imp = IterativeImputer()
df_imputed = pd.DataFrame(imp.fit_transform(df_missing), columns=df.columns)

print('Before:')
print(df_missing.describe())
print('\nAfter:')
print(df_imputed.describe())

## Q2: MAR Pattern + RMSE

In [ ]:
# Generate data
X = np.random.randn(n,4)
X[:,3] = X[:,0]*0.7 + X[:,1]*0.3 + np.random.randn(n)*0.2
df = pd.DataFrame(X, columns=['X1','X2','X3','X4'])

# MAR missing
mask = df['X1'] < df['X1'].median()
true_values = df.loc[mask,'X4'].copy()
df.loc[mask,'X4'] = np.nan

# Impute
imp = IterativeImputer()
df_imputed = pd.DataFrame(imp.fit_transform(df), columns=df.columns)

# RMSE
rmse = np.sqrt(mean_squared_error(true_values, df_imputed.loc[mask,'X4']))
print('RMSE:', rmse)

## Q3: Categorical + Numeric

In [ ]:
# Mixed dataset
df = pd.DataFrame({
    'num1': np.random.randn(n),
    'num2': np.random.randn(n),
    'num3': np.random.randn(n),
    'cat1': np.random.choice(['A','B','C'], n),
    'cat2': np.random.choice(['X','Y'], n)
})

# Introduce missing
df.loc[df.sample(frac=0.2).index, 'cat1'] = np.nan

# Encode
enc = OrdinalEncoder()
df[['cat1','cat2']] = enc.fit_transform(df[['cat1','cat2']])

# Impute
imp = IterativeImputer()
df_imp = pd.DataFrame(imp.fit_transform(df), columns=df.columns)

# Round categorical
df_imp[['cat1','cat2']] = np.round(df_imp[['cat1','cat2']])

print(df_imp.head())

## Q4: Estimator Sensitivity

In [ ]:
estimators = {
    'BayesianRidge': BayesianRidge(),
    'RF': RandomForestRegressor(),
    'ExtraTrees': ExtraTreesRegressor()
}

results = {}
for name, est in estimators.items():
    imp = IterativeImputer(estimator=est)
    X_imp = imp.fit_transform(df_missing.select_dtypes(include=np.number))
    results[name] = np.mean(X_imp)

print(results)

## Q5: Convergence Diagnostics

In [ ]:
iters = [5,10,20]
means = []

for i in iters:
    imp = IterativeImputer(max_iter=i, sample_posterior=True)
    X_imp = imp.fit_transform(df_missing.select_dtypes(include=np.number))
    means.append(np.mean(X_imp))

plt.plot(iters, means)
plt.xlabel('Iterations')
plt.ylabel('Mean Imputed Value')
plt.show()

## Q6: Pipeline + CV

In [ ]:
df['target'] = np.random.randn(n)

X = df.drop('target', axis=1)
y = df['target']

num_cols = ['num1','num2','num3']
cat_cols = ['cat1','cat2']

preprocessor = ColumnTransformer([
    ('num', IterativeImputer(), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder())
    ]), cat_cols)
])

model = Pipeline([
    ('prep', preprocessor),
    ('reg', BayesianRidge())
])

scores = cross_val_score(model, X, y, cv=5)
print('CV Score:', scores.mean())

## Q7: Multiple Imputations

In [ ]:
scores = []
for i in range(5):
    imp = IterativeImputer(sample_posterior=True, random_state=i)
    X_imp = imp.fit_transform(df_missing.select_dtypes(include=np.number))
    scores.append(np.mean(X_imp))

print('Mean:', np.mean(scores), 'Std:', np.std(scores))

## Q8: Real Dataset Test

In [ ]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df_real = data.frame

# Introduce missing
df_real.loc[df_real.sample(frac=0.1).index, 'MedInc'] = np.nan

# Impute
imp = IterativeImputer()
df_imp = pd.DataFrame(imp.fit_transform(df_real), columns=df_real.columns)

print(df_imp.describe())